# Chapter 6 — Validating VaR Models

教材 pp. 74–84；PDF 页序为教材页码加 11。

本章的模型治理、概念合理性、遗漏风险与监管判断是定性内容，不能由一个统计量替代。回测核心复用 Chapter4；这里实现敏感度、区间与基准比较。

下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from validation_calculator import *

敏感度用已知二次函数验证；标准误随样本量平方根下降。

In [2]:
data=pd.DataFrame({'x':[2.,3.]})
estimator=lambda d:(d.x**2).sum()
out=finite_difference_sensitivity(data,'x',estimator(data),estimator,.001)
assert_allclose(out.first_derivative,[4,6])
assert_allclose(out.second_derivative,[2,2],atol=1e-7)
a=quantile_confidence_interval(1.645,.95,.103,100)
b=quantile_confidence_interval(1.645,.95,.103,400)
assert_allclose(a.standard_error,2*b.standard_error)
print(exception_probability_interval(0,250))

lower    0.000000
upper    0.014647
dtype: float64


## 顺序统计量概率区间与基准比较；pp.77–83

In [3]:
interval=order_probability_interval(99,50,.90)
assert_allclose(interval['lower']+interval['upper'],1)
out=benchmark_comparison(pd.DataFrame({'a':[110.],'b':[100.]}),'a','b')
assert_allclose(out['ratio'],1.1)
assert_allclose(out['difference'],10)